# Bitonic Sort using CUDA (Numba) vs CPU

This notebook:
1. Implements Bitonic Sort on GPU using CUDA + Numba.
2. Sorts the same data on CPU using NumPy.
3. Compares execution times.
4. Verifies correctness.


In [ ]:
import numpy as np
import time
from numba import cuda


## CUDA Bitonic Sort Kernel

In [ ]:
@cuda.jit
def bitonic_sort_step(values, j, k):
    i = cuda.grid(1)

    if i < values.size:
        ixj = i ^ j

        if ixj > i and ixj < values.size:

            if (i & k) == 0:
                if values[i] > values[ixj]:
                    temp = values[i]
                    values[i] = values[ixj]
                    values[ixj] = temp
            else:
                if values[i] < values[ixj]:
                    temp = values[i]
                    values[i] = values[ixj]
                    values[ixj] = temp


## Generate Input Data

In [ ]:
N = 1024  # power of 2

arr = np.random.randint(0, 100000, N).astype(np.int32)

cpu_arr = arr.copy()
gpu_arr = arr.copy()

print("Input size:", N)


## CPU Sorting

In [ ]:
start_cpu = time.time()

cpu_sorted = np.sort(cpu_arr)

cpu_time = time.time() - start_cpu

print("CPU Time:", cpu_time, "seconds")


## GPU Bitonic Sort

In [ ]:
d_arr = cuda.to_device(gpu_arr)

threads_per_block = 256
blocks = (N + threads_per_block - 1) // threads_per_block

start_gpu = time.time()

k = 2
while k <= N:

    j = k // 2

    while j > 0:
        bitonic_sort_step[blocks, threads_per_block](d_arr, j, k)
        cuda.synchronize()
        j //= 2

    k *= 2

gpu_sorted = d_arr.copy_to_host()

gpu_time = time.time() - start_gpu

print("GPU Time:", gpu_time, "seconds")


## Verify Correctness

In [ ]:
print("Sorting Correct:", np.array_equal(cpu_sorted, gpu_sorted))


## Performance Comparison

In [ ]:
speedup = cpu_time / gpu_time if gpu_time > 0 else 0

print("CPU Time :", cpu_time)
print("GPU Time :", gpu_time)
print("Speedup  :", speedup)


## Theory

### Bitonic Sort
Bitonic Sort is a parallel sorting algorithm designed for GPU architectures.

### Time Complexity
- CPU Sort (comparison sort): O(n log n)
- Bitonic Sort: O(log² n) stages executed in parallel

### Advantages
- Highly parallel
- Suitable for CUDA/OpenCL
- Deterministic execution pattern

### Applications
- GPU computing
- Scientific simulations
- Real-time graphics
- High-performance computing
